# Hidden-state（按样本均值）与全量梯度分析

已适配最新日志：
- 梯度仅记录 `grad_partition=all`（移除 `_build_partitioned_inputs`）
- hidden state 记录为“**每条样本**在每层的 text token 平均向量 / image token 平均向量

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
sns.set_theme(style='whitegrid', context='talk')


In [ ]:
GRAD_LOG = Path('gradient_logs.jsonl')
HS_LOG = Path('hidden_state_logs.jsonl')
OUT = Path('analysis_outputs')
OUT.mkdir(parents=True, exist_ok=True)

STEP_MIN=None
STEP_MAX=None
STEP_MOD=None
MAX_ROWS=300000
MAX_TSNE=4000
assert GRAD_LOG.exists(), GRAD_LOG
print('grad log:', GRAD_LOG.resolve())
print('hidden log exists:', HS_LOG.exists())


In [ ]:
def step_ok(v):
    try: s=int(v)
    except: return False
    if STEP_MIN is not None and s < STEP_MIN: return False
    if STEP_MAX is not None and s > STEP_MAX: return False
    if STEP_MOD is not None and STEP_MOD > 1 and s % STEP_MOD != 0: return False
    return True

def load_jsonl(path, usecols=None, max_rows=None):
    usecols=set(usecols) if usecols else None
    rows=[]
    if not path.exists():
        return pd.DataFrame()
    with path.open('r',encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: o=json.loads(line)
            except: continue
            if 'step' in o and not step_ok(o['step']):
                continue
            rows.append(o if usecols is None else {k:o.get(k,None) for k in usecols})
            if max_rows and len(rows)>=max_rows: break
    return pd.DataFrame(rows)


## 1) 全量梯度日志（grad_partition=all）

In [ ]:
gdf = load_jsonl(GRAD_LOG, usecols=[
    'step','layer_depth','adapter_type','param_name','grad_partition','grad_norm','grad_norm_per_token',
    'total_supervised_tokens','text_supervised_tokens','image_supervised_tokens','text_token_ratio','image_token_ratio',
    'grad_was_none','grad_path'
], max_rows=MAX_ROWS)

for c in ['step','layer_depth','grad_norm','grad_norm_per_token','text_token_ratio','image_token_ratio']:
    if c in gdf.columns:
        gdf[c]=pd.to_numeric(gdf[c], errors='coerce')
if 'grad_was_none' in gdf.columns:
    gdf['grad_was_none']=gdf['grad_was_none'].fillna(False).astype(bool)

if len(gdf)==0:
    raise ValueError('No gradient data loaded')
print('rows:',len(gdf),'partitions:',sorted(gdf['grad_partition'].dropna().unique().tolist()))


In [ ]:
layer_grad = gdf.groupby('layer_depth', as_index=False).agg(
    mean_grad_norm=('grad_norm','mean'),
    mean_grad_per_token=('grad_norm_per_token','mean'),
    none_ratio=('grad_was_none','mean'),
    mean_img_ratio=('image_token_ratio','mean')
)
layer_grad.to_csv(OUT/'table_layer_all_grad_stats.csv', index=False, encoding='utf-8-sig')

plt.figure(figsize=(11,5))
sns.lineplot(data=layer_grad, x='layer_depth', y='mean_grad_per_token', marker='o')
plt.title('Layer-wise mean grad_norm_per_token (all partition)')
plt.tight_layout(); plt.savefig(OUT/'fig_layer_all_grad_per_token.png', dpi=180); plt.show()

plt.figure(figsize=(11,5))
sns.lineplot(data=layer_grad, x='layer_depth', y='none_ratio', marker='o')
plt.title('Layer-wise none-gradient ratio (all partition)')
plt.tight_layout(); plt.savefig(OUT/'fig_layer_all_none_ratio.png', dpi=180); plt.show()


## 2) Hidden-state 日志（每样本 text/image token 均值向量）

In [ ]:
hs = load_jsonl(HS_LOG, usecols=[
    'step','layer_depth','sample_index','sample_modality_type','modality','token_count','hidden_dim','hidden_path','hidden_norm'
], max_rows=MAX_ROWS)

if len(hs)==0:
    print('No hidden-state logs found. Enable --enable_hidden_state_logging True first.')
else:
    for c in ['step','layer_depth','sample_index','sample_modality_type','token_count','hidden_dim','hidden_norm']:
        if c in hs.columns:
            hs[c]=pd.to_numeric(hs[c], errors='coerce')
    print('hidden rows:',len(hs),'layers:',hs['layer_depth'].nunique())
    display(hs.head(5))


In [ ]:
if len(hs)>0:
    hs_layer = hs.groupby(['layer_depth','modality'], as_index=False).agg(
        mean_hidden_norm=('hidden_norm','mean'),
        std_hidden_norm=('hidden_norm','std'),
        mean_token_count=('token_count','mean')
    )
    hs_layer.to_csv(OUT/'table_layer_hidden_mean_stats.csv', index=False, encoding='utf-8-sig')

    plt.figure(figsize=(11,5))
    sns.lineplot(data=hs_layer, x='layer_depth', y='mean_hidden_norm', hue='modality', marker='o')
    plt.title('Layer-wise hidden norm (sample-level mean vectors)')
    plt.tight_layout(); plt.savefig(OUT/'fig_layer_hidden_norm_modality.png', dpi=180); plt.show()


In [ ]:
def load_vec(path):
    p=Path(path)
    if not p.exists(): return None
    x=np.load(p)
    return x.reshape(-1)

if len(hs)>0:
    layers = sorted(hs['layer_depth'].dropna().unique().tolist())
    pick=[]
    if layers: pick.append(layers[0])
    if len(layers)>2: pick.append(layers[len(layers)//2])
    if len(layers)>1: pick.append(layers[-1])
    pick=sorted(list(set(pick)))
    print('selected layers:', pick)

    for layer in pick:
        sdf=hs[hs['layer_depth']==layer].copy()
        vecs=[]; labs=[]
        for _,r in sdf.iterrows():
            v=load_vec(r['hidden_path'])
            if v is None: continue
            vecs.append(v)
            labs.append(r['modality'])
        if len(vecs)<20:
            continue
        X=np.stack(vecs, axis=0)
        y=np.array(labs)

        # PCA
        Zp=PCA(n_components=2, random_state=42).fit_transform(X)
        pca_df=pd.DataFrame({'x':Zp[:,0],'y':Zp[:,1],'modality':y})
        plt.figure(figsize=(6,5)); sns.scatterplot(data=pca_df,x='x',y='y',hue='modality',s=12,alpha=0.7)
        plt.title(f'Layer {int(layer)} PCA (sample-level mean vectors)')
        plt.tight_layout(); plt.savefig(OUT/f'fig_layer{int(layer)}_pca_hidden_mean.png', dpi=180); plt.show()

        # t-SNE
        Xts, yts = X, y
        if X.shape[0] > MAX_TSNE:
            idx=np.random.choice(X.shape[0], MAX_TSNE, replace=False)
            Xts=X[idx]; yts=y[idx]
        if Xts.shape[0] >= 50:
            Zt=TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto').fit_transform(Xts)
            ts_df=pd.DataFrame({'x':Zt[:,0],'y':Zt[:,1],'modality':yts})
            plt.figure(figsize=(6,5)); sns.scatterplot(data=ts_df,x='x',y='y',hue='modality',s=12,alpha=0.7)
            plt.title(f'Layer {int(layer)} t-SNE (sample-level mean vectors)')
            plt.tight_layout(); plt.savefig(OUT/f'fig_layer{int(layer)}_tsne_hidden_mean.png', dpi=180); plt.show()

        # clustering
        n_clusters=min(4,max(2,len(np.unique(y))))
        km=KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
        cid=km.fit_predict(X)
        sil=silhouette_score(X,cid) if X.shape[0]>n_clusters else np.nan
        ct=pd.crosstab(pd.Series(y,name='modality'), pd.Series(cid,name='cluster'))
        ct.to_csv(OUT/f'table_layer{int(layer)}_cluster_crosstab.csv', encoding='utf-8-sig')
        print(f'layer {int(layer)} silhouette={sil:.4f}, samples={X.shape[0]}')


## 3) 建议

- 你现在的 hidden-state 采集已经是每样本 text/image token 均值，直接适配层空间分布分析。
- 若 text-only 与 text-image 结果仍接近，优先看：
  1) 分层 hidden-space 分离度（silhouette）
  2) 分层 hidden_norm 差异
  3) 全量梯度在高 image_token_ratio 区间的层敏感性。